## AND-105 Task 6: LLM Risk Explanation Prototype

Uses a local LLM (Ollama) to generate natural-language risk explanations for the 10 highest-risk elevators.

**Model chosen:** `llama3.1:8b-instruct-q8_0`  
**Why:** This is the instruct-tuned variant of Llama 3.1 8B at Q8_0 quantization (near full precision). Instruct models follow structured prompts reliably and produce consistent output formats, which matters here because we need data-grounded explanations, not open-ended generation. The 8B size fits in local memory and is large enough for domain-specific reasoning over structured elevator data.

**Note on feature alignment:** The risk scores in the `predictions` table were generated by the ML model in `ml_pipeline.ipynb`. That model's top predictors are `needs_action_rate` (fraction of inspections resulting in 'Needs Action'), `pass_rate`, and `prior_inspection_count`. The context we pass to the LLM includes these same signals — outcome counts and pass rate — so explanations reference the actual features driving each score, not generic observations.

In [1]:
import os
import requests
import psycopg2
import psycopg2.extras
from datetime import datetime, timedelta
from IPython.display import display, Markdown

DB = dict(
    host=os.getenv('POSTGRES_HOST', 'localhost'),
    port=int(os.getenv('POSTGRES_PORT', '5432')),
    dbname=os.getenv('POSTGRES_DB', 'rocket_elevators'),
    user=os.getenv('POSTGRES_USER', 'rocket_user'),
    password=os.getenv('POSTGRES_PASSWORD', 'rocket_pass'),
)
OLLAMA_URL = 'http://localhost:11434/api/chat'
MODEL = 'llama3.1:8b-instruct-q8_0'

conn = psycopg2.connect(**DB)
print('Connected to database')

Connected to database


### Step 1: Query the 10 highest-risk elevators with full context

In [2]:
# Anchor to the dataset's own date range — incidents go up to 2016
with conn.cursor() as _cur:
    _cur.execute("SELECT MAX(date_of_occurrence) FROM incidents")
    _max_date = _cur.fetchone()[0]
TWO_YEARS_AGO = (_max_date - timedelta(days=730)).strftime('%Y-%m-%d')
print(f'Incident window: {TWO_YEARS_AGO} → {_max_date}')

with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:

    # Top 10 by risk_score
    cur.execute("""
        SELECT
            e.elevator_id,
            COALESCE(e.location, 'Unknown')    AS location,
            COALESCE(e.city, 'Unknown')        AS city,
            e.license_status,
            COALESCE(e.device_type, 'Unknown') AS equipment_type,
            p.risk_score,
            p.risk_level
        FROM predictions p
        JOIN elevators e ON e.elevator_id = p.elevator_id
        ORDER BY p.risk_score DESC
        LIMIT 10
    """)
    top10 = [dict(r) for r in cur.fetchall()]
    ids = [e['elevator_id'] for e in top10]

    # Inspection outcome counts (all time) — mirrors needs_action_rate / pass_rate features
    cur.execute("""
        SELECT
            elevator_id,
            COUNT(*)                                           AS total_inspections,
            COUNT(*) FILTER (WHERE outcome = 'Passed')        AS passed_count,
            COUNT(*) FILTER (WHERE outcome != 'Passed')       AS needs_action_count,
            ROUND(
                COUNT(*) FILTER (WHERE outcome = 'Passed')::numeric
                / NULLIF(COUNT(*), 0) * 100, 1
            )                                                  AS pass_rate_pct
        FROM inspections
        WHERE elevator_id = ANY(%s)
        GROUP BY elevator_id
    """, (ids,))
    outcome_stats = {r['elevator_id']: dict(r) for r in cur.fetchall()}

    # Last 5 inspections per elevator
    cur.execute("""
        SELECT elevator_id, inspection_number,
               COALESCE(inspection_type, 'Unknown') AS inspection_type,
               latest_inspection_date::text AS date,
               outcome,
               ROW_NUMBER() OVER (PARTITION BY elevator_id ORDER BY latest_inspection_date DESC) AS rn
        FROM inspections
        WHERE elevator_id = ANY(%s)
    """, (ids,))
    insp_rows = [dict(r) for r in cur.fetchall()]
    inspections_by_id = {i: [] for i in ids}
    for row in insp_rows:
        if row['rn'] <= 5:
            inspections_by_id[row['elevator_id']].append(row)

    # Incidents in the most recent 2 years of the dataset
    cur.execute("""
        SELECT elevator_id,
               COALESCE(category, 'Unknown') AS category,
               date_of_occurrence::text      AS date,
               COALESCE(root_cause, '')      AS root_cause
        FROM incidents
        WHERE elevator_id = ANY(%s)
          AND date_of_occurrence >= %s
    """, (ids, TWO_YEARS_AGO))
    incident_rows = [dict(r) for r in cur.fetchall()]
    incidents_by_id = {i: [] for i in ids}
    for row in incident_rows:
        incidents_by_id[row['elevator_id']].append(row)

    # Recent alterations
    cur.execute("""
        SELECT elevator_id,
               COALESCE(alteration_type, 'Unknown') AS alteration_type,
               COALESCE(status, '')                 AS status
        FROM alterations
        WHERE elevator_id = ANY(%s)
    """, (ids,))
    alteration_rows = [dict(r) for r in cur.fetchall()]
    alterations_by_id = {i: [] for i in ids}
    for row in alteration_rows:
        alterations_by_id[row['elevator_id']].append(row)

# Merge everything into one record per elevator
elevators = []
for e in top10:
    eid = e['elevator_id']
    stats = outcome_stats.get(eid, {})
    elevators.append({
        **e,
        'total_inspections':  stats.get('total_inspections', 0),
        'passed_count':       stats.get('passed_count', 0),
        'needs_action_count': stats.get('needs_action_count', 0),
        'pass_rate_pct':      stats.get('pass_rate_pct', None),
        'inspections': inspections_by_id[eid],
        'incidents':   incidents_by_id[eid],
        'alterations': alterations_by_id[eid],
    })

print(f'Loaded {len(elevators)} elevators')
for e in elevators:
    pct = f"{e['pass_rate_pct']}%" if e['pass_rate_pct'] is not None else 'n/a'
    print(f"  #{e['elevator_id']} score={e['risk_score']:.4f} ({e['risk_level']}) | "
          f"{e['total_inspections']} insp ({pct} pass rate) | "
          f"{len(e['incidents'])} incidents | {len(e['alterations'])} alterations")

Incident window: 2014-11-23 → 2016-11-22
Loaded 10 elevators
  #73700 score=0.9924 (high) | 3 insp (0.0% pass rate) | 0 incidents | 0 alterations
  #72362 score=0.9924 (high) | 2 insp (0.0% pass rate) | 0 incidents | 0 alterations
  #19443 score=0.9920 (high) | 3 insp (33.3% pass rate) | 0 incidents | 1 alterations
  #21305 score=0.9919 (high) | 6 insp (0.0% pass rate) | 0 incidents | 2 alterations
  #4401 score=0.9919 (high) | 4 insp (0.0% pass rate) | 0 incidents | 1 alterations
  #64463 score=0.9917 (high) | 3 insp (33.3% pass rate) | 0 incidents | 1 alterations
  #72216 score=0.9915 (high) | 4 insp (25.0% pass rate) | 0 incidents | 0 alterations
  #33385 score=0.9915 (high) | 4 insp (50.0% pass rate) | 0 incidents | 1 alterations
  #2684 score=0.9910 (high) | 4 insp (0.0% pass rate) | 0 incidents | 0 alterations
  #28462 score=0.9904 (high) | 5 insp (20.0% pass rate) | 0 incidents | 1 alterations


### Step 2: Ollama helper

In [3]:
def call_ollama(system_prompt: str, user_message: str) -> str:
    response = requests.post(OLLAMA_URL, json={
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user',   'content': user_message},
        ],
        'stream': False,
        'options': {'temperature': 0.2},
    })
    response.raise_for_status()
    return response.json()['message']['content'].strip()


def elevator_to_text(e: dict) -> str:
    pct = f"{e['pass_rate_pct']}%" if e['pass_rate_pct'] is not None else 'unknown'
    lines = [
        f"Elevator ID: {e['elevator_id']}",
        f"Location: {e['location']}, {e['city']}",
        f"Equipment type: {e['equipment_type']}",
        f"License status: {e['license_status']}",
        f"Risk score: {e['risk_score']:.4f} ({e['risk_level']} risk)",
        "",
        "Inspection outcome summary (all time):",
        f"  Total: {e['total_inspections']}  |  Passed: {e['passed_count']}  |  Needs Action: {e['needs_action_count']}  |  Pass rate: {pct}",
        "",
        "Last 5 inspections (most recent first):",
    ]
    if e['inspections']:
        for insp in e['inspections']:
            lines.append(f"  - {insp['date']} | {insp['inspection_type']} | outcome: {insp['outcome']}")
    else:
        lines.append('  - No inspection records')

    lines.append("")
    lines.append("Incidents (most recent 2 years of dataset):")
    if e['incidents']:
        for inc in e['incidents']:
            lines.append(f"  - {inc['date']} | {inc['category']} | {inc['root_cause']}")
    else:
        lines.append('  - None')

    lines.append("")
    lines.append("Recent alterations:")
    if e['alterations']:
        for alt in e['alterations'][:3]:
            lines.append(f"  - {alt['alteration_type']} | status: {alt['status']}")
    else:
        lines.append('  - None')

    return '\n'.join(lines)

print('Helper functions ready')

Helper functions ready


### Step 3: System prompt — Version 1 (baseline)

**Design rationale:** Defines a role, specifies a 2-3 sentence output format, provides domain context (Ontario elevator compliance), and instructs the model to cite specific data points. Primary risk factors listed explicitly match the top ML features: inspection outcome rate, incident history, license status.

In [4]:
SYSTEM_V1 = """You are a licensed elevator safety analyst working for the Ontario Ministry of Transportation.
Your task is to write a 2-3 sentence risk explanation for a specific elevator based on its compliance history.

The risk score is driven primarily by: (1) the fraction of inspections resulting in 'Needs Action',
(2) incident history, and (3) license status. Lead with whichever of these three is most severe.

Rules:
- Cite specific dates, counts, and pass rates from the data provided. Do not make generic statements.
- Only reference facts explicitly present in the data above.
- If a data category is empty, omit it from the explanation entirely.
- Output only the explanation. No headers, no bullet points."""

print('System prompt V1 defined')
print(SYSTEM_V1)

System prompt V1 defined
You are a licensed elevator safety analyst working for the Ontario Ministry of Transportation.
Your task is to write a 2-3 sentence risk explanation for a specific elevator based on its compliance history.

The risk score is driven primarily by: (1) the fraction of inspections resulting in 'Needs Action',
(2) incident history, and (3) license status. Lead with whichever of these three is most severe.

Rules:
- Cite specific dates, counts, and pass rates from the data provided. Do not make generic statements.
- Only reference facts explicitly present in the data above.
- If a data category is empty, omit it from the explanation entirely.
- Output only the explanation. No headers, no bullet points.


### Step 4: Generate explanations for all 10 elevators (V1 prompt)

In [5]:
results_v1 = []

for e in elevators:
    user_msg = elevator_to_text(e)
    explanation = call_ollama(SYSTEM_V1, user_msg)
    results_v1.append({'elevator': e, 'explanation': explanation})
    display(Markdown(
        f"---\n**Elevator {e['elevator_id']}** — score {e['risk_score']:.4f} ({e['risk_level']}) "
        f"| pass rate {e['pass_rate_pct']}%\n\n"
        f"*{e['location']}, {e['city']} | {e['equipment_type']}*\n\n"
        f"{explanation}"
    ))

print(f'Generated {len(results_v1)} explanations')

---
**Elevator 73700** — score 0.9924 (high) | pass rate 0.0%

*278 Manitoba St, Toronto, Toronto | Observation Elevator*

Elevator 73700 at 278 Manitoba St, Toronto poses a high risk due to its perfect record of failing all three inspections conducted since 2013, with no passes and three 'Needs Action' outcomes. This is the primary driver of its high risk score. The elevator's license status remains active despite this history.

---
**Elevator 72362** — score 0.9924 (high) | pass rate 0.0%

*C N E (Ntc) Automotive Tunnel, Toronto, Toronto | Passenger Elevator*

Elevator 72362 at the C N E (Ntc) Automotive Tunnel in Toronto poses a high risk due to its consistently poor inspection history, with all two inspections resulting in 'Needs Action' and a corresponding pass rate of 0.0%. This is further exacerbated by a lack of incident data within the past two years, indicating that any potential issues have not yet manifested as safety incidents.

---
**Elevator 19443** — score 0.9920 (high) | pass rate 33.3%

*37 Lasalle Blvd, Sudbury, Sudbury | Passenger Elevator*

Elevator 19443 at 37 Lasalle Blvd, Sudbury poses a high risk due to its poor compliance history, with only one passing inspection out of three total inspections (33.3% pass rate), and two requiring action. The most recent inspection on January 22, 2016, initially required follow-up but was later resolved.

---
**Elevator 21305** — score 0.9919 (high) | pass rate 0.0%

*33 Dawson Rd, Guelph, Guelph | Passenger Elevator*

Elevator 21305 at 33 Dawson Rd, Guelph poses a high risk due to its alarming inspection history, with all six inspections resulting in 'Needs Action' and a pass rate of 0.0%. This is particularly concerning given the most recent inspection on April 24, 2015, which required follow-up action.

---
**Elevator 4401** — score 0.9919 (high) | pass rate 0.0%

*635 Queen St E, Toronto, Toronto | Freight Elevator*

Elevator 4401 at 635 Queen St E, Toronto poses a high risk due to its consistently poor inspection history, with all four inspections resulting in 'Needs Action' and a pass rate of 0.0%. Despite recent completion of an ED-Minor A Alteration, the elevator's underlying safety issues remain unaddressed.

...

---
**Elevator 72216** — score 0.9915 (high) | pass rate 25.0%

*2 Market St, Toronto, Toronto | Passenger Elevator*

Elevator 72216 at 2 Market St, Toronto poses a high risk due to its extremely poor inspection compliance history, with only one passing inspection out of four total inspections, resulting in a pass rate of just 25.0%. This is further exacerbated by the fact that three consecutive inspections have resulted in 'Needs Action' outcomes since 2013, indicating ongoing safety concerns.

---
**Elevator 33385** — score 0.9915 (high) | pass rate 50.0%

*85 West St, Goderich, Goderich | Passenger Elevator*

Elevator 33385 at 85 West St, Goderich poses a high risk due to its poor compliance history, with only 50% pass rate across all inspections and two 'Needs Action' outcomes out of four total inspections. The elevator's license is ACTIVE, but this does not mitigate the significant risk posed by its inspection record.

---
**Elevator 2684** — score 0.9910 (high) | pass rate 0.0%

*Brownstock Washer, Espanola, Espanola | Freight Elevator*

Elevator 2684 at the Brownstock Washer in Espanola poses a high risk due to its consistently poor inspection history, with all four inspections resulting in 'Needs Action' and a pass rate of 0% over time. The most recent inspection on April 9, 2015, required follow-up action, indicating ongoing issues that have not been addressed.

---
**Elevator 28462** — score 0.9904 (high) | pass rate 20.0%

*695 Regency Crt, Burlington, Burlington | Passenger Elevator*

Elevator 28462 at 695 Regency Crt, Burlington poses a high risk due to its extremely poor compliance history, with only one passing inspection out of five total inspections, resulting in a pass rate of just 20.0%. This is further exacerbated by the fact that four inspections required "Needs Action" outcomes, indicating significant safety concerns.

Generated 10 explanations


### Step 5: Prompt variations on elevators 1, 2, 3

Testing 3 prompt directions on the same 3 elevators to compare output quality. V2 and V3 were developed using `/branch` from the same context as V1.

**V2 — Strict guardrails** (branch `prompt-guardrails`): Tightens rules against filler and speculation. Removes causal priming, adds explicit ban on commenting about missing data, and merges the conflicting cite/omit rules.  
**V3 — Contrast forcing** (branch `prompt-contrast`): Forces the LLM to identify what makes each elevator unique among high-risk elevators instead of producing a generic low-pass-rate explanation.

In [8]:
SYSTEM_V2 = """You are a licensed elevator safety analyst under Ontario Regulation 209/01 (Elevating Devices).
Risk scores range from 0.0 (no risk) to 1.0 (critical). Scores above 0.85 indicate imminent compliance action.
The score is driven primarily by inspection outcome history — a high 'Needs Action' rate is the strongest single predictor.

Write a 2-3 sentence explanation of why this elevator has a high risk score.

Rules:
- Only reference facts explicitly present in the data. Never infer, speculate, or explain absences.
- If a data category is empty, omit it entirely. Do not comment on missing data.
- Cite specific counts, pass rates, and dates.
- Output only the explanation. No headers, no bullet points."""

SYSTEM_V3 = """You are a licensed elevator safety analyst under Ontario Regulation 209/01 (Elevating Devices).
Risk scores range from 0.0 (no risk) to 1.0 (critical). Scores above 0.85 indicate imminent compliance action.
The score is driven primarily by inspection outcome history — a high 'Needs Action' rate is the strongest
single predictor. Incidents in the past 2 years are weighted more heavily than older records.

Write a 2-3 sentence explanation of why this elevator has a high risk score.

Your explanation MUST identify what is specific to this elevator's situation — not just its pass rate.
Ask yourself: what combination of factors makes this elevator stand out even among other high-risk elevators?
Consider: equipment type, location context, number of inspections, recency of failures, incident or alteration history.
Only reference facts explicitly present in the data. If a data category is empty, omit it entirely.
Output only the explanation."""

test_elevators = elevators[:3]

print("=== PROMPT V2 (richer domain context) ===")
for e in test_elevators:
    explanation = call_ollama(SYSTEM_V2, elevator_to_text(e))
    display(Markdown(f"**Elevator {e['elevator_id']}** (V2)\n\n{explanation}"))

print("\n=== PROMPT V3 (contrast forcing) (V3) ===")
for e in test_elevators:
    explanation = call_ollama(SYSTEM_V3, elevator_to_text(e))
    display(Markdown(f"**Elevator {e['elevator_id']}** (contrast)\n\n{explanation}"))


=== PROMPT V2 (richer domain context) ===


**Elevator 73700** (V2)

This elevator has a high risk score due to its perfect record of failing all three inspections, with no passes and a pass rate of 0.0%, dating back to at least 2013. The most recent inspection on November 11, 2016, also resulted in a follow-up action, indicating ongoing safety concerns.

**Elevator 72362** (V2)

This elevator has a high risk score due to its perfect record of failing all two inspections, with no passes and two "Needs Action" outcomes since the start of inspection records. The most recent inspection on January 28, 2015, also resulted in a follow-up action, indicating ongoing issues that have not been addressed.

**Elevator 19443** (V2)

Elevator 19443 has a high risk score due to its poor inspection history, with only one pass out of three inspections and a pass rate of 33.3% over all time. The most recent two inspections in 2016 resulted in "Needs Action" outcomes before being resolved on the follow-up inspection.


=== PROMPT V3 (contrast forcing) (V3) ===


**Elevator 73700** (contrast)

Elevator 73700 has a high risk score due to its consistent failure to pass inspections, with all three inspections resulting in "Needs Action" outcomes. This is particularly concerning given the elevator's observation status and active license status, suggesting that it should be functioning properly. The lack of recent incidents or alterations does not mitigate this risk, as the repeated inspection failures indicate a persistent issue with the elevator's safety.

**Elevator 72362** (contrast)

Elevator 72362 at the C N E (Ntc) Automotive Tunnel in Toronto has a high risk score due to its perfect record of "Needs Action" inspection outcomes, with no inspections passing over its entire history. This is particularly concerning given that it has undergone two periodic inspections within the past decade, yet failed to meet safety standards on both occasions.

**Elevator 19443** (contrast)

Elevator 19443 at 37 Lasalle Blvd, Sudbury has a high risk score due to its extremely low pass rate of 33.3% across only three inspections, with two requiring follow-up action. Notably, the most recent inspection was an unscheduled ED inspection followed by a follow-up inspection that resolved all orders, indicating ongoing issues despite a minor alteration being passed recently.

### Step 6: Comparison and observations

| Prompt | Strengths | Weaknesses |
|--------|-----------|------------|
| V1 (baseline) | Cites real numbers (pass rate, counts). Follows the 2-3 sentence rule. | Generates filler when incident data is empty. Speculates about resolutions not present in the data. |
| V2 (strict guardrails) | Cites specific dates. Eliminates filler for empty incident categories. More concise. | Still speculates in some cases ("ongoing issues that have not been addressed", "before being resolved"). |
| V3 (contrast forcing) | Mentions equipment type and differentiating factors. More specific per elevator. | Introduces hallucinations under contrast pressure ("shut down twice" — not in the data). |

**Chosen approach:** V2 (strict guardrails) — best balance between precision and filler elimination. V3 generates specificity but at the cost of hallucinations, which is unacceptable in a compliance context. V2 reduces speculation without sacrificing citation of concrete data points.

### Step 7: Writer/Reviewer — system prompt review

The Writer session produced V2 (strict guardrails) as the best prompt after /branch exploration.
A fresh Reviewer session with no implementation context reviewed V2 for hallucination triggers, ambiguous instructions, and missing guardrails.

**Findings from the Reviewer session:**

1. **Hallucination trigger — hardcoded framing:** "Write a 2-3 sentence explanation of why this elevator has a high risk score" asserts the score is high before the model sees the data. The model will rationalize a high-risk narrative even for a moderate score. Fix: gate the framing on the actual score value.

2. **Causal priming:** "The score is driven primarily by inspection outcome history" trains the model to lead with inspections even when the data shows something else (e.g., a recent incident with no inspections). The model fits the data to match the stated causal theory instead of reading the data first. Fix: remove this sentence and let the data drive the explanation.

3. **Contradictory rules:** "Cite specific counts, pass rates, and dates" conflicts with "If a data category is empty, omit it entirely." If no dates or counts are present, the model must violate one rule to satisfy the other — and may fabricate values. Fix: merge them — "Cite specific counts, pass rates, and dates where present. Omit any category for which no value is available."

4. **No ban on invented numbers:** "Never infer or speculate" does not explicitly cover the model inventing a plausible-sounding count (e.g., "3 of 5 inspections failed"). Fix: add "Do not write any number, date, or count that does not appear verbatim in the data."

**What the Reviewer surfaced that the Writer missed:**

The hardcoded framing issue (#1) was the most significant blind spot. The Writer session felt the prompt was complete because V2 performed better than V1 in testing. The Reviewer came in without that context and immediately spotted that the instruction structure assumes the conclusion before reading the evidence — a classic hallucination setup. The Writer was too focused on fixing the filler problem from V1 to notice the framing issue in the task instruction itself.

### Final prompt (post-Reviewer fixes)

```
You are a data analyst summarizing elevator risk indicators for compliance review.
Risk scores range from 0.0 (no risk) to 1.0 (critical). Scores above 0.85 indicate imminent compliance action.

Write 1-3 sentences summarizing the key risk indicators for this elevator based on its score of {score}.
Use only as many sentences as the data supports. Do not repeat or rephrase facts to meet a length requirement.

Rules:
- Begin with the actual score value: "With a risk score of {score}..."
- Cite specific counts, pass rates, and dates where present. Omit any category for which no value is available.
- Do not write any number, date, or count that does not appear verbatim in the data. Do not round, estimate, or approximate.
- Do not reference regulatory sections, clauses, or legal obligations not present in the data.
- Output only the explanation. No headers, no bullet points.
```

In [ ]:
conn.close()
print('Done')